In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier,plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,ConfusionMatrixDisplay

In [ ]:
#generate larger farming dataset
np.random.seed(42)
n_samples=60
data={
    'weather':np.random.choice(['sunny','cloudy','rainy'],n_samples),
    'soil':np.random.choice(['dry','moist','wet'],n_samples),
    'temperature':np.random.choice(['hot','mild','cool'],n_samples),
    'humidity':np.random.choice(['high','medium','low'],n_samples),
    'wind':np.random.choice(['strong','weak'],n_samples),
    'ferilizer':np.random.choice(['yes','no'],n_samples),
    'croptype':np.random.choice(['corn','wheat','rice'],n_samples),
    'season':np.random.choice(['kharif','rabi','summer'],n_samples)
}
df=pd.DataFrame(data)



In [ ]:
#create target variable (rule-based for realism)
def irrigation_rule(row):
  if row['soil']=='dry' and row['weather']!='rainy':
    return 'Yes'
  elif row['soil']=='wet':
    return 'No'
  elif row['humidity']=='high' and row['weather']=='rainy':
    return 'No'
  else:
    return np.random.choice(['Yes','No'])
df['irrigate']=df.apply(irrigation_rule,axis=1)
print("Sample Dataset:\n",df.head())

Sample Dataset:
   weather   soil temperature humidity    wind ferilizer croptype  season  \
0   rainy  moist         hot     high    weak       yes     rice    rabi   
1   sunny    dry        cool      low    weak       yes    wheat    rabi   
2   rainy  moist        mild   medium  strong       yes    wheat  kharif   
3   rainy  moist        cool     high  strong       yes    wheat  kharif   
4   sunny  moist        cool     high    weak        no    wheat  kharif   

  irrigate  
0       No  
1      Yes  
2       No  
3       No  
4       No  


In [ ]:
#encode categorical varables
le_dict={}
for column in  df.columns:
  le=LabelEncoder()
  df[column]=le.fit_transform(df[column])
  le_dict[column]=le
print("Encoded Dataset:\n",df.head())

Encoded Dataset:
    weather  soil  temperature  humidity  wind  ferilizer  croptype  season  \
0        1     1            1         0     1          1         1       1   
1        2     0            0         1     1          1         2       1   
2        1     1            2         2     0          1         2       0   
3        1     1            0         0     0          1         2       0   
4        2     1            0         0     1          0         2       0   

   irrigate  
0         0  
1         1  
2         0  
3         0  
4         0  


In [ ]:
#split features and target
x=df.drop('irrigate',axis=1)
y=df['irrigate']

In [ ]:
#train-test split(stratified)
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42,stratify=y)

In [ ]:
#decision tree model
dt_model=DecisionTreeClassifier(criterion='entropy',random_state=42)
dt_model.fit(x_train,y_train)
y_pred_dt=dt_model.predict(x_test)

In [ ]:
#random forest model
rf_model=RandomForestClassifier(
    n_estimators=20,
    criterion='entropy',
    random_state=42
)
rf_model.fit(x_train,y_train)
y_pred_rf=rf_model.predict(x_test)

In [ ]:
#accuracy evaluation
dt_accuracy=accuracy_score(y_test,y_pred_dt)
rf_accuracy=accuracy_score(y_test,y_pred_rf)
print("\nAccuracy Comparison:")
print("Decision Tree Accuracy:",dt_accuracy)
print("Random Forest Accuracy:",rf_accuracy)



Accuracy Comparison:
Decision Tree Accuracy: 0.7777777777777778
Random Forest Accuracy: 0.7777777777777778


In [ ]:
#confusion matrix(FIXED+MEANINGFUL)
cm_dt=confusion_matrix(y_test,y_pred_dt,labels=[0,1])
cm_rf=confusion_matrix(y_test,y_pred_rf,labels=[0,1])
cm_dt_df=pd.DataFrame(cm_dt,index=['Actual No','Actual Yes'],columns=['Predicted No','Predicted Yes'])
cm_rf_df=pd.DataFrame(cm_rf,index=['Actual No','Actual Yes'],columns=['Predicted No','Predicted Yes'])
print("\nDecision Tree Confusion Matrix:\n,cm_dt_df")
print("\nRandom Forest Confusion Matrix:\n",cm_rf_df)


Decision Tree Confusion Matrix:
,cm_dt_df

Random Forest Confusion Matrix:
             Predicted No  Predicted Yes
Actual No              9              2
Actual Yes             2              5


In [2]:
from sklearn.metrics import ConfusionMatrixDisplay

#confusion matrix visualization
ConfusionMatrixDisplay.from_predictions(y_test,y_pred_dt,labels=[0,1])
plt.title("Decision Tree Confusion Matrix")
plt.show()
ConfusionMatrixDisplay.from_predictions(y_test,y_pred_rf,labels=[0,1])
plt.title("Random Forest Confusion Matrix")
plt.show()

NameError: name 'y_test' is not defined

In [ ]:
#cross validation
dt_cv=cross_val_score(dt_model,x,y,cv=5)
rf_cv=cross_val_score(rf_model,x,y,cv=5)
print("\nCross Validation Scores:")
print("Decision Tree :",dt_cv.mean())
print("Random Forest:",rf_cv.mean())



Cross Validation Scores:
Decision Tree : 0.7833333333333334
Random Forest: 0.8


In [ ]:
#feature importance(random forest)
importance=pd.DataFrame({
    'feature':x.columns,
    'importance':rf_model.feature_importances_
}).sort_values(by='importance',ascending=False)
print("\nFeature Importance:\n",importance)



Feature Importance:
        feature  importance
1         soil    0.353088
3     humidity    0.156683
0      weather    0.115385
7       season    0.089851
5    ferilizer    0.083963
4         wind    0.073617
2  temperature    0.068440
6     croptype    0.058972
